# 🎵 ĐỒ ÁN MÔN HỌC: TƯƠNG TÁC DỮ LIỆU (INTERACTIVE DATA VISUALIZATION)
## PHẦN 1: THU THẬP, LÀM SẠCH & TIỀN XỬ LÝ DỮ LIỆU ĐA NGUỒN (01_preprocess.ipynb)

> **Nhóm thực hiện:** Nhóm nghiên cứu Xu hướng Âm nhạc & Kinh tế Vĩ mô  
> **Người phụ trách chính:** Thành viên 1 (Data Engineer / Pipeline Specialist)  
> **Barem đánh giá:** Mục 1 — Thu thập và Tiền xử lý dữ liệu (2.5 điểm)

---

### 📌 1. Xác định bài toán & Giới thiệu 5 Bộ Dữ liệu (0.5 điểm)
- **Mục tiêu phân tích:** Làm rõ mối quan hệ giữa các **đặc trưng âm học (Audio Features)** (*Danceability*, *Energy*, *Valence*, *Tempo*...) với hành vi nghe nhạc thịnh hành trên Spotify, đồng thời đánh giá tác động của **tiềm lực kinh tế quốc gia (GDP, nhóm thu nhập)** đến thị hiếu âm nhạc đại chúng.
- **Quy mô dữ liệu:** Đạt **50,000 dòng** sau tiền xử lý và lọc phân tầng đại diện trên nền tảng dữ liệu sạch từ hơn 1.69 triệu bản ghi bảng xếp hạng gốc (vượt xa yêu cầu $\ge$ 5,000 dòng của barem).
- **Hệ thống 5 bảng dữ liệu đa nguồn:**
  1. `spotify-weekly-charts.csv` (~1.78 triệu dòng, 36 cột): Bảng xếp hạng tuần tại 70+ quốc gia và Global (2020–2022).
  2. `spotify-tracks-dataset.csv` (~114,000 bài hát, 22 cột): Danh mục bài hát và các chỉ số âm học chi tiết từ Spotify API.
  3. `gdp_data.csv` (13,365 bản ghi): Tổng sản phẩm quốc nội (GDP quy đổi USD) chuỗi thời gian của Ngân hàng Thế giới (World Bank).
  4. `country_codes.csv` (217 quốc gia): Bảng phân loại khu vực địa lý (*Region*) và nhóm thu nhập (*Income Group*).
  5. `best-selling-artists.csv` (121 nghệ sĩ): Danh sách nghệ sĩ bán đĩa chạy nhất mọi thời đại kèm thể loại và doanh số.

---

### 🛡️ Bảng giải quyết 3 lỗi thiết kế nghiêm trọng & Pre-flight Checklist

| Mã | Vấn đề kỹ thuật | Rủi ro nếu bỏ qua | Giải pháp đã khắc phục triệt để |
| :---: | :--- | :--- | :--- |
| **Lỗi 1** | Sampling 2 lớp chồng nhau (cắt 3.200 dòng/chunk) | Mất cân bằng tỷ lệ quốc gia, dữ liệu không đại diện | **Bỏ hoàn toàn việc sample khi đọc chunk.** Đọc full 100% dữ liệu 2020-2022 (~1.69M dòng). Chỉ thực hiện Stratified Sampling 1 lần duy nhất ở cuối pipeline. |
| **Lỗi 2** | Lọc outlier `duration` không tác động đến dataset cuối | Dataset vẫn chứa bài hát có thời lượng dị biệt | Áp dụng bộ lọc `30,000 <= duration <= 1,200,000` ms (30s - 20 phút) **trực tiếp trên `df_charts['duration']`** ngay sau khi ép kiểu số. |
| **Lỗi 3** | Cột `popularity` bị giả lập bằng hàm tuyến tính từ `rank` | Target leakage nghiêm trọng, làm sai lệch mô hình & tương quan | **Xóa bỏ hoàn toàn công thức giả định.** Tạo `popularity_real` (giữ nguyên NaN nếu không khớp) và `has_real_popularity` (flag nhị phân). |
| **P1** | Lệch tên quốc gia giữa Charts và GDP | Mất dữ liệu khi JOIN | Xây dựng bảng từ điển ánh xạ đầy đủ 7 trường hợp lệch; đạt tỷ lệ match **98.6%** (72/73 quốc gia). |
| **P2** | Dòng `country == 'Global'` trong Charts | Sai logic thống kê khi gộp với GDP quốc gia | Tách riêng toàn bộ 41,879 dòng `Global` vào `df_global`, loại khỏi phân tích cấp quốc gia. |
| **P8** | Chuỗi doanh số dính liền trong `best-selling-artists` | Bóc tách sai doanh số | Xây dựng biểu thức chính quy (Regex) trích xuất số đầu tiên trước từ khóa "million". |


In [1]:
# 1. KHỞI TẠO MÔI TRƯỜNG & DATA PROFILING (KIỂM TRA CẤU TRÚC 5 FILE GỐC)
import os
import re
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if not os.path.exists(os.path.join(BASE_DIR, 'data')):
    BASE_DIR = r"d:\uni\hk1nam3\idvi\idvi-final_project\Spotify_Audio_Trends_Analytics"

DATA_RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

print("=" * 70)
print("1.1. PROFILING 4 BẢNG DỮ LIỆU ĐỘC LẬP")
print("=" * 70)

# A. Best-selling artists
df_artists_raw = pd.read_csv(os.path.join(DATA_RAW_DIR, 'best-selling-artists.csv'))
print(f"- best-selling-artists: shape = {df_artists_raw.shape}, missing values = {df_artists_raw.isnull().sum().to_dict()}")

# B. Country codes
df_cc_raw = pd.read_csv(os.path.join(DATA_RAW_DIR, 'country_codes.csv'))
print(f"- country_codes: shape = {df_cc_raw.shape}, unique codes = {df_cc_raw['country_code'].nunique()}")

# C. GDP data
df_gdp_raw = pd.read_csv(os.path.join(DATA_RAW_DIR, 'gdp_data.csv'))
print(f"- gdp_data: shape = {df_gdp_raw.shape}, years range = {df_gdp_raw['year'].min()} - {df_gdp_raw['year'].max()}")

# D. Spotify tracks dataset
df_tracks_raw = pd.read_csv(os.path.join(DATA_RAW_DIR, 'spotify-tracks-dataset.csv'))
print(f"- spotify-tracks-dataset: shape = {df_tracks_raw.shape}, unique track_id = {df_tracks_raw['track_id'].nunique():,}")

print("\n" + "=" * 70)
print("1.2. PROFILING BẢNG CHÍNH: SPOTIFY WEEKLY CHARTS (~800MB)")
print("=" * 70)
charts_preview = pd.read_csv(os.path.join(DATA_RAW_DIR, 'spotify-weekly-charts.csv'), nrows=10, low_memory=False)
print("Định dạng cột 'uri' (Xác nhận P4):")
for uri in charts_preview['uri'].head(5):
    print(f"  {uri} -> track_id trích xuất: {uri.split(':')[-1]}")

print("\nCăn cứ lựa chọn khung thời gian 2020-2022 (Xác nhận P7):")
print("- Giai đoạn 2016 - 2019: Spotify chỉ theo dõi vài quốc gia nhỏ, chỉ có ~18,000 bản ghi/năm.")
print("- Giai đoạn 2020 - 2022: Mở rộng lên 70+ quốc gia toàn cầu với hơn 1.73 triệu bản ghi (>97% tập dữ liệu),")
print("  đồng thời bao trùm trọn vẹn thời kỳ biến động xã hội của đại dịch COVID-19.")


1.1. PROFILING 4 BẢNG DỮ LIỆU ĐỘC LẬP
- best-selling-artists: shape = (121, 7), missing values = {'Artist name': 0, 'Country': 0, 'Active years': 0, 'Release year of first charted record': 0, 'Genre': 0, 'Total certified units': 0, 'Claimed sales': 0}
- country_codes: shape = (217, 3), unique codes = 217
- gdp_data: shape = (13365, 4), years range = 1960 - 2022


- spotify-tracks-dataset: shape = (114000, 22), unique track_id = 89,741

1.2. PROFILING BẢNG CHÍNH: SPOTIFY WEEKLY CHARTS (~800MB)
Định dạng cột 'uri' (Xác nhận P4):
  spotify:track:2gpQi3hbcUAcEG8m2dlgfB -> track_id trích xuất: 2gpQi3hbcUAcEG8m2dlgfB
  spotify:track:2x8oBuYaObjqHqgGuIUZ0b -> track_id trích xuất: 2x8oBuYaObjqHqgGuIUZ0b
  spotify:track:2SJZdZ5DLtlRosJ2xHJJJa -> track_id trích xuất: 2SJZdZ5DLtlRosJ2xHJJJa
  spotify:track:1O2pcBJGej0pmH2Y9XZMs6 -> track_id trích xuất: 1O2pcBJGej0pmH2Y9XZMs6
  spotify:track:1TpZKxGnHp37ohJRszTSiq -> track_id trích xuất: 1TpZKxGnHp37ohJRszTSiq

Căn cứ lựa chọn khung thời gian 2020-2022 (Xác nhận P7):
- Giai đoạn 2016 - 2019: Spotify chỉ theo dõi vài quốc gia nhỏ, chỉ có ~18,000 bản ghi/năm.
- Giai đoạn 2020 - 2022: Mở rộng lên 70+ quốc gia toàn cầu với hơn 1.73 triệu bản ghi (>97% tập dữ liệu),
  đồng thời bao trùm trọn vẹn thời kỳ biến động xã hội của đại dịch COVID-19.


In [2]:
# 2. LÀM SẠCH DỮ LIỆU TỪNG BẢNG (DATA CLEANING - 0.5 ĐIỂM)

# ------------------------------------------------------------------------------
# 2.1. Xử lý bảng best-selling-artists & Kiểm thử Regex bóc tách doanh số (P8)
# ------------------------------------------------------------------------------
def parse_sales(val):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip()
    m = re.search(r'([0-9]+(?:\.[0-9]+)?)\s*million', val_str, re.IGNORECASE)
    if m:
        return float(m.group(1))
    m_any = re.search(r'([0-9]+(?:\.[0-9]+)?)', val_str)
    return float(m_any.group(1)) if m_any else 0.0

# Kiểm thử 5 Test Cases cụ thể (bao gồm trường hợp dính liền 2 con số)
test_cases = [
    '600 million500 million',
    '250 million',
    '144.28 millionUS: 67.050 million',
    '75 million[83]',
    '289.5 millionUS: 177.3 millionJPN: 4.650 million'
]
print("Kiểm thử thuật toán bóc tách doanh số (P8 Test Cases):")
for tc in test_cases:
    print(f"  [Input]: '{tc}' --> [Output]: {parse_sales(tc)} triệu bản")

df_artists = df_artists_raw.copy()
df_artists['sales_millions'] = df_artists['Claimed sales'].apply(parse_sales)
df_artists['clean_artist_name'] = df_artists['Artist name'].astype(str).str.lower().str.strip()
legend_artists_set = set(df_artists['clean_artist_name'].unique())

print("\nBảng đối chiếu 5 nghệ sĩ hàng đầu sau khi bóc tách doanh số:")
print(df_artists[['Artist name', 'Claimed sales', 'sales_millions']].head())

# ------------------------------------------------------------------------------
# 2.2. Xử lý GDP, Country Codes & Xây dựng Từ điển Ánh xạ Quốc gia (P1, P2)
# ------------------------------------------------------------------------------
df_gdp_clean = df_gdp_raw.rename(columns={'value': 'gdp_usd'})
df_gdp_full = pd.merge(df_gdp_clean, df_cc_raw, on='country_code', how='left')

# Bảng mapping chuẩn hóa tên quốc gia giữa Spotify Charts và World Bank
country_mapping = {
    'Czech Republic': 'Czechia',
    'Egypt': 'Egypt, Arab Rep.',
    'Hong Kong': 'Hong Kong SAR, China',
    'Korea': 'Korea, Rep.',
    'Slovakia': 'Slovak Republic',
    'Turkey': 'Turkiye',
    'Venezuela': 'Venezuela, RB'
}

# ------------------------------------------------------------------------------
# 2.3. Chuẩn bị bảng Tracks (spotify-tracks-dataset)
# ------------------------------------------------------------------------------
# Lấy các trường bổ sung: track_genre, popularity, explicit
# Đổi tên popularity thành popularity_real để phản ánh tính trung thực dữ liệu (Sửa Lỗi 3)
df_tracks_sub = df_tracks_raw[['track_id', 'track_genre', 'popularity', 'explicit']].drop_duplicates(subset=['track_id'])
df_tracks_sub = df_tracks_sub.rename(columns={'popularity': 'popularity_real'})
print(f"\nBảng phụ trợ tracks trích xuất: {df_tracks_sub.shape[0]:,} bài hát duy nhất.")


Kiểm thử thuật toán bóc tách doanh số (P8 Test Cases):
  [Input]: '600 million500 million' --> [Output]: 600.0 triệu bản
  [Input]: '250 million' --> [Output]: 250.0 triệu bản
  [Input]: '144.28 millionUS: 67.050 million' --> [Output]: 144.28 triệu bản
  [Input]: '75 million[83]' --> [Output]: 75.0 triệu bản
  [Input]: '289.5 millionUS: 177.3 millionJPN: 4.650 million' --> [Output]: 289.5 triệu bản

Bảng đối chiếu 5 nghệ sĩ hàng đầu sau khi bóc tách doanh số:
       Artist name           Claimed sales  sales_millions
0      The Beatles  600 million500 million         600.000
1  Michael Jackson  500 million400 million         500.000
2    Elvis Presley             500 million         500.000
3       Elton John  300 million250 million         300.000
4            Queen  300 million250 million         300.000

Bảng phụ trợ tracks trích xuất: 89,741 bài hát duy nhất.


In [3]:
# 2.4. NẠP CHUNKS CHARTS KHÔNG SAMPLING SỚM (SỬA LỖI 1) & LỌC OUTLIER DURATION (SỬA LỖI 2)
print("Đang nạp toàn bộ dữ liệu Weekly Charts theo Chunking (KHÔNG sample sớm, giữ trọn vẹn dữ liệu 2020-2022)...")

use_cols = [
    'uri', 'rank', 'artist_names', 'artist_individual', 'artist_genre',
    'track_name', 'release_date', 'peak_rank', 'previous_rank',
    'weeks_on_chart', 'streams', 'week', 'country',
    'danceability', 'energy', 'key', 'mode', 'loudness',
    'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration'
]

chunks = []
global_chunks = []
total_scanned = 0
header_rows_count = 0

t0 = time.time()
for chunk in pd.read_csv(os.path.join(DATA_RAW_DIR, 'spotify-weekly-charts.csv'), chunksize=200000, low_memory=False, usecols=use_cols):
    total_scanned += len(chunk)
    # Loại bỏ dòng lặp header rác
    is_header = (chunk['rank'] == 'rank')
    header_rows_count += is_header.sum()
    clean = chunk[~is_header].copy()
    
    # Lọc khung thời gian nghiên cứu 2020-2022 (P7)
    clean = clean[clean['week'] >= '2020-01-01']
    
    # Tách riêng toàn bộ dòng Global (P2)
    is_global = (clean['country'] == 'Global')
    global_part = clean[is_global]
    country_part = clean[~is_global]
    
    if len(global_part) > 0:
        global_chunks.append(global_part) # Giữ TOÀN BỘ Global, không sample 500 dòng như cũ
    if len(country_part) > 0:
        chunks.append(country_part)       # Giữ TOÀN BỘ country chunks, không sample 3,200 dòng

df_charts = pd.concat(chunks, ignore_index=True)
df_global = pd.concat(global_chunks, ignore_index=True)

print(f"-> Tổng số dòng quét từ file gốc: {total_scanned:,}")
print(f"-> Số dòng header rác bị loại bỏ: {header_rows_count:,}")
print(f"-> Tổng số dòng df_charts (cấp quốc gia) trước khi JOIN: {len(df_charts):,} dòng (TĂNG MẠNH từ 57.6k cũ!)")
print(f"-> Tổng số dòng df_global (toàn cầu lưu riêng): {len(df_global):,} dòng")
print(f"-> Thời gian nạp: {time.time() - t0:.2f} giây")

# Ép kiểu dữ liệu số
numeric_fields = ['rank', 'streams', 'peak_rank', 'previous_rank', 'weeks_on_chart',
                  'danceability', 'energy', 'key', 'mode', 'loudness',
                  'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration']
for col in numeric_fields:
    df_charts[col] = pd.to_numeric(df_charts[col], errors='coerce')

# Loại bỏ giá trị khuyết thiếu cốt lõi
df_charts = df_charts.dropna(subset=['rank', 'streams', 'danceability', 'energy']).copy()
df_charts['rank'] = df_charts['rank'].astype(int)

# ------------------------------------------------------------------------------
# LỌC OUTLIER DURATION TRỰC TIẾP TRÊN DF_CHARTS (SỬA LỖI 2)
# ------------------------------------------------------------------------------
len_dur_before = len(df_charts)
# Ngưỡng: từ 30 giây (30,000 ms) đến 20 phút (1,200,000 ms)
df_charts = df_charts[(df_charts['duration'] >= 30000) & (df_charts['duration'] <= 1200000)].copy()
len_dur_after = len(df_charts)
dur_outliers_removed = len_dur_before - len_dur_after

print(f"\n🛡️ [SỬA LỖI 2] Lọc Outlier Duration trực tiếp trên df_charts:")
print(f"-> Số bản ghi bị loại bỏ vì thời lượng dị biệt (<30s hoặc >20 phút): {dur_outliers_removed:,} dòng")
print(f"-> Số bản ghi df_charts hợp lệ còn lại: {len(df_charts):,} dòng")

# Chuẩn hóa thời gian & Trích xuất thuộc tính thời vụ (P3)
df_charts['week_date'] = pd.to_datetime(df_charts['week'], errors='coerce')
df_charts['year'] = df_charts['week_date'].dt.year
df_charts['month'] = df_charts['week_date'].dt.month
df_charts['quarter'] = df_charts['week_date'].dt.to_period('Q').astype(str)

def map_season(m):
    if m in [3, 4, 5]:
        return 'Xuân (Spring)'
    elif m in [6, 7, 8]:
        return 'Hạ (Summer)'
    elif m in [9, 10, 11]:
        return 'Thu (Autumn)'
    else:
        return 'Đông (Winter)'

df_charts['season'] = df_charts['month'].apply(map_season)

# Trích xuất track_id từ uri (P4)
df_charts['track_id'] = df_charts['uri'].astype(str).str.split(':').str[-1]

# Ánh xạ tên quốc gia (P1)
df_charts['country_clean'] = df_charts['country'].replace(country_mapping)

chart_countries = set(df_charts['country_clean'].unique())
gdp_countries = set(df_gdp_full['country_name'].unique())
matched = chart_countries.intersection(gdp_countries)
print(f"\nKết quả kiểm tra tỷ lệ khớp quốc gia với GDP (P1 Match Rate):")
print(f"-> Khớp thành công: {len(matched)} / {len(chart_countries)} quốc gia ({len(matched)/len(chart_countries)*100:.2f}%)")


Đang nạp toàn bộ dữ liệu Weekly Charts theo Chunking (KHÔNG sample sớm, giữ trọn vẹn dữ liệu 2020-2022)...


-> Tổng số dòng quét từ file gốc: 1,787,999
-> Số dòng header rác bị loại bỏ: 73
-> Tổng số dòng df_charts (cấp quốc gia) trước khi JOIN: 1,691,994 dòng (TĂNG MẠNH từ 57.6k cũ!)
-> Tổng số dòng df_global (toàn cầu lưu riêng): 41,879 dòng
-> Thời gian nạp: 10.72 giây



🛡️ [SỬA LỖI 2] Lọc Outlier Duration trực tiếp trên df_charts:
-> Số bản ghi bị loại bỏ vì thời lượng dị biệt (<30s hoặc >20 phút): 25 dòng
-> Số bản ghi df_charts hợp lệ còn lại: 1,691,574 dòng



Kết quả kiểm tra tỷ lệ khớp quốc gia với GDP (P1 Match Rate):
-> Khớp thành công: 72 / 73 quốc gia (98.63%)


In [4]:
# 3. KẾT NỐI 5 BẢNG DỮ LIỆU & TẠO CÁC TRƯỜNG TÍNH TOÁN (SỬA LỖI 3 - NO FAKE POPULARITY)
print("Đang thực hiện JOIN 5 bảng trên toàn bộ 1.69 triệu dòng dữ liệu sạch...")

t_join = time.time()
# Bước 1: Join Charts với GDP & Country Codes theo [country_clean, year]
df_merged = pd.merge(
    df_charts,
    df_gdp_full[['country_name', 'country_code', 'year', 'gdp_usd', 'region', 'income_group']],
    left_on=['country_clean', 'year'],
    right_on=['country_name', 'year'],
    how='left'
)

# Bước 2: Join với bảng spotify-tracks-dataset để lấy thể loại và thuộc tính bổ sung
df_merged = pd.merge(df_merged, df_tracks_sub, on='track_id', how='left')
df_merged['track_genre'] = df_merged['track_genre'].fillna(df_merged['artist_genre']).fillna('pop')
df_merged['explicit'] = df_merged['explicit'].fillna(0).astype(int)

# ------------------------------------------------------------------------------
# CHUẨN HÓA 1,710 THỂ LOẠI NGÁCH VỀ 12 MAJOR GENRES (GIẢI QUYẾT PHÂN MẢNH THỂ LOẠI)
# ------------------------------------------------------------------------------
def map_major_genre(genre_raw):
    g = str(genre_raw).lower().strip()
    if g in ['0', 'unknown', 'none', 'nan', '']:
        return 'Other / Unknown'
    
    if any(k in g for k in ['latin', 'reggaeton', 'latino', 'urbano', 'ranchera', 'corrido', 'salsa', 'bachata', 'cumbia', 'mariachi', 'flamenco', 'sertanejo', 'pagode', 'forro', 'funk rj', 'funk bh', 'cuarteto', 'mambo', 'merengue']):
        return 'Latin & Reggaeton'
    elif any(k in g for k in ['hip-hop', 'hip hop', 'rap', 'trap', 'drill', 'grime', 'boom bap']):
        return 'Hip-Hop & Rap'
    elif any(k in g for k in ['k-pop', 'kpop', 'j-pop', 'jpop', 'c-pop', 'cpop', 'v-pop', 'vpop']):
        return 'East Asian Pop (K-Pop/J-Pop)'
    elif any(k in g for k in ['rock', 'punk', 'metal', 'grunge', 'hard rock', 'emo']):
        return 'Rock & Metal'
    elif any(k in g for k in ['edm', 'electronic', 'house', 'techno', 'trance', 'dance', 'electro', 'dubstep', 'dnb', 'drum and bass', 'club']):
        return 'Dance & Electronic'
    elif any(k in g for k in ['r&b', 'rnb', 'soul', 'funk', 'neo soul', 'motown']):
        return 'R&B & Soul'
    elif any(k in g for k in ['indie', 'alternative', 'folk', 'acoustic', 'singer-songwriter']):
        return 'Indie & Alternative'
    elif any(k in g for k in ['pop']):
        return 'Pop'
    elif any(k in g for k in ['country', 'bluegrass', 'americana']):
        return 'Country'
    elif any(k in g for k in ['jazz', 'blues', 'classical', 'orchestral', 'opera']):
        return 'Jazz & Classical'
    elif any(k in g for k in ['reggae', 'afrobeats', 'afropop', 'dancehall']):
        return 'Afrobeats & Reggae'
    else:
        return 'Other / Regional'

df_merged['major_genre'] = df_merged['track_genre'].apply(map_major_genre)
print(f"-> Chuẩn hóa thể loại: từ {df_merged['track_genre'].nunique()} micro-genres về {df_merged['major_genre'].nunique()} Major Genres cốt lõi!")

# ------------------------------------------------------------------------------
# XỬ LÝ POPULARITY THỰC CHẤT (SỬA LỖI 3 - LOẠI BỎ TARGET LEAKAGE)
# ------------------------------------------------------------------------------
# KHÔNG dùng công thức giả lập: 100 - int(r/2)
# Giữ nguyên popularity_real từ tracks-dataset (để NaN nếu không khớp)
# Tạo cờ has_real_popularity để phân biệt minh bạch khi phân tích
df_merged['has_real_popularity'] = df_merged['popularity_real'].notna().astype(int)
num_real_pop = df_merged['has_real_popularity'].sum()

print(f"🛡️ [SỬA LỖI 3] Xử lý Minh bạch Popularity (Tránh Target Leakage):")
print(f"-> Số bản ghi có popularity THẬT từ Spotify API: {num_real_pop:,} / {len(df_merged):,} ({num_real_pop/len(df_merged)*100:.2f}%)")
print(f"-> Số bản ghi để NaN (không khớp): {len(df_merged) - num_real_pop:,} ({100 - num_real_pop/len(df_merged)*100:.2f}%)")

# Bước 3: Đánh dấu nghệ sĩ bán đĩa chạy nhất lịch sử (Best-selling artists)
clean_artists = df_merged['artist_individual'].fillna(df_merged['artist_names']).astype(str).str.lower().str.strip()
def match_legend(name):
    if name in legend_artists_set:
        return 1
    parts = re.split(r'[,;&/]', str(name))
    for p in parts:
        if p.strip() in legend_artists_set:
            return 1
    return 0

df_merged['is_best_selling_artist'] = clean_artists.apply(match_legend)

# ------------------------------------------------------------------------------
# 3.2. Tạo 8 Calculated Fields có ý nghĩa phân tích sâu
# ------------------------------------------------------------------------------
# 1. log_streams: Chuẩn hóa logarit cho lượt stream (giảm độ lệch cực đại)
df_merged['log_streams'] = np.log1p(df_merged['streams'])

# 2. is_top10: Nhãn nhị phân cho các siêu bản hit dẫn đầu
df_merged['is_top10'] = (df_merged['rank'] <= 10).astype(int)

# 3. is_top50: Nhãn nhị phân cho bài hát thịnh hành
df_merged['is_top50'] = (df_merged['rank'] <= 50).astype(int)

# 4. stream_volume_tier: Phân tầng 4 cấp độ quy mô bài hát
def assign_tier(rank):
    if rank <= 10:
        return 'Mega Hit (Top 10)'
    elif rank <= 50:
        return 'Major Hit (Top 11-50)'
    elif rank <= 100:
        return 'Moderate Hit (Top 51-100)'
    else:
        return 'Emerging Hit (Top 101-200)'

df_merged['stream_volume_tier'] = df_merged['rank'].apply(assign_tier)

# 5. mood_cluster: Phân loại 4 góc phần tư cảm xúc dựa trên Valence x Energy (ngưỡng 0.5)
mood_conds = [
    (df_merged['valence'] >= 0.5) & (df_merged['energy'] >= 0.5),
    (df_merged['valence'] < 0.5) & (df_merged['energy'] >= 0.5),
    (df_merged['valence'] >= 0.5) & (df_merged['energy'] < 0.5),
    (df_merged['valence'] < 0.5) & (df_merged['energy'] < 0.5)
]
mood_names = [
    'Vui tươi & Hào hứng (Happy/Energetic)',
    'Kịch tính & Sôi động (Intense/Party)',
    'Bình yên & Thư thái (Chill/Peaceful)',
    'Trầm lắng & U sầu (Sad/Melancholic)'
]
df_merged['mood_cluster'] = np.select(mood_conds, mood_names, default='Chưa phân loại')

# 6. duration_min: Quy đổi thời lượng sang phút (đã được lọc outlier)
df_merged['duration_min'] = df_merged['duration'] / 60000.0

# 7. is_new_entry: Đánh dấu ca khúc mới ra mắt lọt bảng xếp hạng
df_merged['is_new_entry'] = df_merged['previous_rank'].isna().astype(int)
df_merged['previous_rank'] = df_merged['previous_rank'].fillna(0).astype(int)

# 8. energy_dance_score: Chỉ số tổng hợp năng động âm học
df_merged['energy_dance_score'] = (df_merged['energy'] + df_merged['danceability']) / 2.0

print(f"-> JOIN & Calculated Fields hoàn tất trong {time.time() - t_join:.2f}s!")
print(f"-> Kích thước tập dữ liệu hợp nhất: {len(df_merged):,} dòng, {df_merged.shape[1]} cột")
print(f"-> Số lượt xuất hiện của nghệ sĩ bán đĩa chạy nhất: {df_merged['is_best_selling_artist'].sum():,}")


Đang thực hiện JOIN 5 bảng trên toàn bộ 1.69 triệu dòng dữ liệu sạch...


-> Chuẩn hóa thể loại: từ 1811 micro-genres về 13 Major Genres cốt lõi!
🛡️ [SỬA LỖI 3] Xử lý Minh bạch Popularity (Tránh Target Leakage):
-> Số bản ghi có popularity THẬT từ Spotify API: 532,352 / 1,691,574 (31.47%)
-> Số bản ghi để NaN (không khớp): 1,159,222 (68.53%)


-> JOIN & Calculated Fields hoàn tất trong 6.91s!
-> Kích thước tập dữ liệu hợp nhất: 1,691,574 dòng, 51 cột
-> Số lượt xuất hiện của nghệ sĩ bán đĩa chạy nhất: 130,441


In [5]:
# 4. STRATIFIED SAMPLING 50,000 DÒNG TRÊN NỀN DỮ LIỆU ĐẠI DIỆN & XUẤT CSV
print("Đang tiến hành Stratified Sampling 50,000 dòng theo [country, year, track_genre] trên toàn bộ 1.69M dòng...")

t_sample = time.time()
target_rows = 50000
frac = min(1.0, target_rows / len(df_merged))

grouped = df_merged.groupby(['country', 'year', 'track_genre'], group_keys=False)
sampled_records = []
small_group_count = 0

for name, grp in grouped:
    n_grp = len(grp)
    if n_grp <= 3:
        small_group_count += 1
        sampled_records.append(grp)
    else:
        take_n = max(2, int(np.ceil(n_grp * frac)))
        sampled_records.append(grp.sample(n=min(take_n, n_grp), random_state=42))

df_final = pd.concat(sampled_records, ignore_index=True)
if len(df_final) > target_rows:
    df_final = df_final.sample(n=target_rows, random_state=42).reset_index(drop=True)

print(f"-> Stratified Sampling hoàn tất trong {time.time() - t_sample:.2f} giây.")

# Danh sách cột tinh gọn chuẩn hóa (thêm major_genre, thay thế popularity bằng popularity_real + has_real_popularity)
cols_order = [
    'track_id', 'uri', 'track_name', 'artist_names', 'artist_individual', 'release_date',
    'track_genre', 'major_genre', 'popularity_real', 'has_real_popularity', 'explicit',
    'rank', 'previous_rank', 'peak_rank', 'weeks_on_chart', 'streams', 'log_streams',
    'week', 'year', 'month', 'quarter', 'season', 'is_top10', 'is_top50', 'is_new_entry', 'stream_volume_tier',
    'country', 'country_name', 'country_code', 'region', 'income_group', 'gdp_usd',
    'danceability', 'energy', 'key', 'mode', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_min',
    'mood_cluster', 'energy_dance_score', 'is_best_selling_artist'
]
df_final = df_final[cols_order].copy()

# ==============================================================================
# KIỂM CHỨNG TOÀN VẸN VÀ BÁO CÁO KẾT QUẢ ĐẦU RA
# ==============================================================================
print("=" * 70)
print("BÁO CÁO KIỂM CHỨNG TOÀN VẸN DỮ LIỆU ĐẦU RA (POST-FIX VERIFICATION):")
print("=" * 70)
print(f"1. Kích thước tập dữ liệu cuối: {df_final.shape[0]:,} dòng × {df_final.shape[1]} cột")
print(f"2. Số lượng quốc gia đại diện: {df_final['country'].nunique()} quốc gia")
print(f"3. Số lượng micro-genres: {df_final['track_genre'].nunique()} | Major Genres chuẩn hóa: {df_final['major_genre'].nunique()}")
print("\nPhân bổ 12 Major Genres trong tập mẫu 50,000 dòng:")
for g_name, g_cnt in df_final['major_genre'].value_counts().items():
    print(f"    + {g_name:30s}: {g_cnt:5,d} bài ({g_cnt/len(df_final)*100:5.2f}%)")
print("\n4. Kiểm tra Outliers Duration (duration_min.describe()):")
print(df_final['duration_min'].describe())

print("\n5. Kiểm tra Tỷ lệ Popularity Thật (has_real_popularity):")
print(df_final['has_real_popularity'].value_counts(normalize=True))

print("\n6. Phân bổ quy mô mẫu theo Nhóm Thu nhập (World Bank):")
for grp, cnt in df_final['income_group'].value_counts(dropna=False).items():
    print(f"    + {grp}: n = {cnt:,}")

# Xuất dữ liệu ra file data/processed/merged_spotify_data.csv
out_path = os.path.join(DATA_PROCESSED_DIR, 'merged_spotify_data.csv')
df_final.to_csv(out_path, index=False, encoding='utf-8')
file_mb = os.path.getsize(out_path) / (1024 * 1024)

print(f"\n🎉 XUẤT FILE THÀNH CÔNG: {out_path} ({file_mb:.2f} MB)")
print("=" * 70)


Đang tiến hành Stratified Sampling 50,000 dòng theo [country, year, track_genre] trên toàn bộ 1.69M dòng...


-> Stratified Sampling hoàn tất trong 28.48 giây.
BÁO CÁO KIỂM CHỨNG TOÀN VẸN DỮ LIỆU ĐẦU RA (POST-FIX VERIFICATION):
1. Kích thước tập dữ liệu cuối: 50,000 dòng × 47 cột
2. Số lượng quốc gia đại diện: 73 quốc gia
3. Số lượng micro-genres: 1710 | Major Genres chuẩn hóa: 13

Phân bổ 12 Major Genres trong tập mẫu 50,000 dòng:
    + Hip-Hop & Rap                 : 11,806 bài (23.61%)
    + Pop                           : 8,711 bài (17.42%)
    + Latin & Reggaeton             : 7,130 bài (14.26%)
    + Other / Regional              : 6,828 bài (13.66%)
    + Dance & Electronic            : 5,197 bài (10.39%)
    + Rock & Metal                  : 2,751 bài ( 5.50%)
    + Indie & Alternative           : 2,263 bài ( 4.53%)
    + R&B & Soul                    : 1,861 bài ( 3.72%)
    + Other / Unknown               : 1,724 bài ( 3.45%)
    + East Asian Pop (K-Pop/J-Pop)  : 1,086 bài ( 2.17%)
    + Afrobeats & Reggae            :   252 bài ( 0.50%)
    + Country                       :   215 bà


🎉 XUẤT FILE THÀNH CÔNG: D:\uni\hk1nam3\idvi\idvi-final_project\Spotify_Audio_Trends_Analytics\data\processed\merged_spotify_data.csv (23.51 MB)
